# DKL-BEACON-stem-edx-live-mic
Prepared by [Utkarsh Pratiush](https://github.com/utkarshp1161)
- [parent notebook link where this is tested on Digital twin microscope](https://github.com/utkarshp1161/Active-learning-in-microscopy/blob/main/notebooks/single_objective_BO_SVDKL-novelty-search.ipynb)



- also note that -- edx
    - dispersion - currentlyl 20ev / channel
    - sum - over enire counts

In [ ]:
import numpy as np
from pathlib import Path
import random
from datetime import datetime
import pickle
import matplotlib.pyplot as plt

## 3. Single Objective Bayesian optimization with DKL

### 3a. DKL model 

In [ ]:
import gpytorch
from botorch.posteriors.gpytorch import GPyTorchPosterior
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy
import math
import torch.nn as nn
import numpy as np
from typing import Tuple, Optional, Dict, Union, List
import numpy as np
import torch

# Simple ConvNet for feature extraction
class ConvNetFeatureExtractor(nn.Module):
    def __init__(self, input_channels=1, output_dim=32):
        super(ConvNetFeatureExtractor, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.output_dim = output_dim
        self.fc = None  # Placeholder for the fully connected layer

    def forward(self, x):
        if len(x.shape) == 3: # TODO: hacky way to make sure botorch acquisition function works
            # flatten
            batch_size, channel, mn = x.shape[0], x.shape[1] , x.shape[2]
            d = math.sqrt(mn)      ## TODO: what if mn is not a perfect square?
            x = x.reshape(int(batch_size), int(channel), int(d), int(d))
        # Pass through the convolutional layers
        x = self.conv_layers(x)


        # If the fully connected layer is not defined yet, initialize it dynamically******************key
        if self.fc is None:
            flattened_size = x.view(x.size(0), -1).size(1)
            device = x.device# TODO: better way to handle device
            self.fc = nn.Linear(flattened_size, self.output_dim).to(device)  # Create fc layer on the correct device

        # Flatten for fully connected layer
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


# GP model with deep kernel using ConvNet feature extractor
class GPModelDKL(ApproximateGP):
    def __init__(self, inducing_points, likelihood, feature_extractor=None):
        if feature_extractor is None:
            feature_extractor = ConvNetFeatureExtractor(
                input_channels=1,  # Set according to your image channels
                output_dim=32      # Set as per the final feature dimension
            ).to(inducing_points.device)
        else:
            feature_extractor = feature_extractor.to(inducing_points.device)

        # Transform inducing points with ConvNet
        inducing_points = feature_extractor(inducing_points)

        # Variational setup
        variational_distribution = CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )

        super(GPModelDKL, self).__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
        self.num_outputs = 1  # must be one
        self.likelihood = likelihood
        self.feature_extractor = feature_extractor

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)



    def __call__(self, x, use_feature_extractor=True, *args, **kwargs):
        ## TODO: to make it compatible with botorch acquisition function we need it to make patches internally from flattened patches
        if use_feature_extractor:
            if len(x.shape) == 3:
                # flatten
                batch_size, channel, mn = x.shape[0], x.shape[1] , x.shape[2]
                d = math.sqrt(mn)      ## TODO: what if mn is not a perfect square?
                x = x.reshape(int(batch_size), int(channel), int(d), int(d))
            x = self.feature_extractor(x)
        return super().__call__(x, *args, **kwargs)

    def posterior(self, X, output_indices=None, observation_noise=False, *args, **kwargs) -> GPyTorchPosterior:
        self.eval()
        self.likelihood.eval()
        dist = self.likelihood(self(X))
        return GPyTorchPosterior(dist)

    @property
    def hparam_dict(self):
        return {
            "likelihood.noise": self.likelihood.noise.item(),
            "covar_module.base_kernel.outputscale": self.covar_module.base_kernel.outputscale.item(),
            "mean_module.constant": self.mean_module.constant.item(),
        }




### 3b. Utility F:n's - 1

In [ ]:
def normalize_data(data : np.ndarray) -> np.ndarray:  # Expected data type: torch.Tensor
    """Normalize data to the [0, 1] range."""
    return (data - data.min()) / (data.max() - data.min())


def numpy_to_torch_for_conv(np_array) -> torch.Tensor:
    """
    Converts a NumPy array of shape (batch_size, a, b) to a PyTorch tensor
    with shape (batch_size, 1, a, b) for neural network use.

    Parameters:
        np_array (np.ndarray): Input NumPy array of shape (batch_size, a, b).

    Returns:
        torch.Tensor: Converted PyTorch tensor of shape (batch_size, 1, a, b).
    """
    # Check if input is a numpy array
    if not isinstance(np_array, np.ndarray):
        raise TypeError("Input must be a NumPy array.")

    # Convert to PyTorch tensor and add a channel dimension
    tensor = torch.from_numpy(np_array).float()  # Convert to float tensor
    tensor = tensor.unsqueeze(1)  # Add a channel dimension at index 1

    return tensor


######################atomai utils####################################
#Credits Maxim Ziatdinov (https://github.com/ziatdinovmax): https://github.com/pycroscopy/atomai/blob/8db3e944cd9ece68c33c8e3fcca3ef3ce9a111ea/atomai/utils/img.py#L522

def get_coord_grid(imgdata: np.ndarray, step: int,
                   return_dict: bool = True
                   ) -> Union[np.ndarray, Dict[int, np.ndarray]]:
    """
    Generate a square coordinate grid for every image in a stack. Returns coordinates
    in a dictionary format (same format as generated by atomnet.predictor)
    that can be used as an input for utility functions extracting subimages
    and atomstat.imlocal class

    Args:
        imgdata (numpy array): 2D or 3D numpy array
        step (int): distance between grid points
        return_dict (bool): returns coordiantes as a dictionary (same format as atomnet.predictor)

    Returns:
        Dictionary or numpy array with coordinates
    """
    if np.ndim(imgdata) == 2:
        imgdata = np.expand_dims(imgdata, axis=0)
    coord = []
    for i in range(0, imgdata.shape[1], step):
        for j in range(0, imgdata.shape[2], step):
            coord.append(np.array([i, j]))
    coord = np.array(coord)
    if return_dict:
        coord = np.concatenate((coord, np.zeros((coord.shape[0], 1))), axis=-1)
        coordinates_dict = {i: coord for i in range(imgdata.shape[0])}
        return coordinates_dict
    coordinates = [coord for _ in range(imgdata.shape[0])]
    return np.concatenate(coordinates, axis=0)

def get_imgstack(imgdata: np.ndarray,
                 coord: np.ndarray,
                 r: int) -> Tuple[np.ndarray]:
    """
    Extracts subimages centered at specified coordinates
    for a single image

    Args:
        imgdata (3D numpy array):
            Prediction of a neural network with dimensions
            :math:`height \\times width \\times n channels`
        coord (N x 2 numpy array):
            (x, y) coordinates
        r (int):
            Window size

    Returns:
        2-element tuple containing

        - Stack of subimages
        - (x, y) coordinates of their centers
    """
    img_cr_all = []
    com = []
    for c in coord:
        cx = int(np.around(c[0]))
        cy = int(np.around(c[1]))
        if r % 2 != 0:
            img_cr = np.copy(
                imgdata[cx-r//2:cx+r//2+1,
                        cy-r//2:cy+r//2+1])
        else:
            img_cr = np.copy(
                imgdata[cx-r//2:cx+r//2,
                        cy-r//2:cy+r//2])
        if img_cr.shape[0:2] == (int(r), int(r)) and not np.isnan(img_cr).any():
            img_cr_all.append(img_cr[None, ...])
            com.append(c[None, ...])
    if len(img_cr_all) == 0:
        return None, None
    img_cr_all = np.concatenate(img_cr_all, axis=0)
    com = np.concatenate(com, axis=0)
    return img_cr_all, com

def extract_subimages(imgdata: np.ndarray,
                      coordinates: Union[Dict[int, np.ndarray], np.ndarray],
                      window_size: int, coord_class: int = 0) -> Tuple[np.ndarray]:
    """
    Extracts subimages centered at certain atom class/type
    (usually from a neural network output)

    Args:
        imgdata (numpy array):
            4D stack of images (n, height, width, channel).
            It is also possible to pass a single 2D image.
        coordinates (dict or N x 2 numpy arry): Prediction from atomnet.locator
            (can be from other source but must be in the same format)
            Each element is a :math:`N \\times 3` numpy array,
            where *N* is a number of detected atoms/defects,
            the first 2 columns are *xy* coordinates
            and the third columns is class (starts with 0).
            It is also possible to pass N x 2 numpy array if the corresponding
            imgdata is a single 2D image.
        window_size (int):
            Side of the square for subimage cropping
        coord_class (int):
            Class of atoms/defects around around which the subimages
            will be cropped (3rd column in the atomnet.locator output)

    Returns:
        3-element tuple containing

        - stack of subimages,
        - (x, y) coordinates of their centers,
        - frame number associated with each subimage
    """
    if isinstance(coordinates, np.ndarray):
        coordinates = np.concatenate((
            coordinates, np.zeros((coordinates.shape[0], 1))), axis=-1)
        coordinates = {0: coordinates}
    if np.ndim(imgdata) == 2:
        imgdata = imgdata[None, ..., None]
    subimages_all, com_all, frames_all = [], [], []
    for i, (img, coord) in enumerate(
            zip(imgdata, coordinates.values())):
        coord_i = coord[np.where(coord[:, 2] == coord_class)][:, :2]
        stack_i, com_i = get_imgstack(img, coord_i, window_size)
        if stack_i is None:
            continue
        subimages_all.append(stack_i)
        com_all.append(com_i)
        frames_all.append(np.ones(len(com_i), int) * i)
    if len(subimages_all) > 0:
        subimages_all = np.concatenate(subimages_all, axis=0)
        com_all = np.concatenate(com_all, axis=0)
        frames_all = np.concatenate(frames_all, axis=0)
    return subimages_all, com_all, frames_all

### 3c. Utility F:n's - 2

In [ ]:
#*********************************DTmic specific functions starts **********************************************#
from sklearn.metrics import mean_squared_error

# setup edx acquisition
import autoscript_tem_toolkit.vision as vision_toolkit
from autoscript_tem_microscope_client.structures import RunOptiStemSettings, RunStemAutoFocusSettings, Point, StagePosition, AdornedImage, EdsAcquisitionSettings, AdornedSpectrum,  StemAcquisitionSettings, StageVelocity, EdsSpectrumImageSettings
from autoscript_tem_microscope_client.enumerations import DetectorType, CameraType, OptiStemMethod, OpticalMode, EdsDetectorType, ExposureTimeType




def configure_acquisition(exposure_time=2):
    """Configure the EDS acquisition settings."""
    # mic_server is global variable intriduced in def run function
    microscope = mic_server
    eds_detector_name = microscope.detectors.eds_detectors[0]
    eds_detector = microscope.detectors.get_eds_detector(eds_detector_name)
    # Configure the acquisition
    global eds_settings
    eds_settings = EdsAcquisitionSettings()
    eds_settings.eds_detector = eds_detector_name
    eds_settings.dispersion = eds_detector.dispersions[-1]
    eds_settings.shaping_time = eds_detector.shaping_times[-1]
    eds_settings.exposure_time = exposure_time
    eds_settings.exposure_time_type = ExposureTimeType.LIVE_TIME
    return eds_settings

def get_channel_index(energy_keV: float, dispersion: float, offset: float) -> int:
    """Convert energy (keV) into spectrum channel index."""
    return int(round((energy_keV - offset) / dispersion))

import xmltodict
import json
import numpy as np

def get_dispersion_and_offset(spectrum):
    """
    Extract dispersion and offset from EDS spectrum metadata (xml).
    Returns (dispersion_keV_per_ch, offset_keV).
    """
    xml_string = spectrum.metadata.metadata_as_xml
    metadata = xmltodict.parse(xml_string)
    metadata = json.loads(json.dumps(metadata))

    detectors = metadata["Metadata"]["Detectors"]["AnalyticalDetector"]

    # If only one detector, wrap it into a list
    if isinstance(detectors, dict):
        detectors = [detectors]

    # Take the first detector (or filter by name if needed)
    det = detectors[0]
    dispersion = float(det.get("Dispersion", 0))
    offset = float(det.get("OffsetEnergy", 0))

    return dispersion, offset


def get_eds_black_box(index, indices_all, e1a, e1b, eds_settings, image_size, element="Zr") -> float:
    # 1) Move beam
    x = int(indices_all[index, 1]) / image_size
    y = int(indices_all[index, 0]) / image_size
    print("collecting spectrum at fractional coord", x, y, "true coord", x*image_size, y*image_size)
    mic_server.optics.paused_scan_beam_position = [x, y]

    # 2) Acquire EDS spectrum
    mic_server.optics.unblank()
    spectrum = mic_server.analysis.eds.acquire_spectrum(eds_settings)
    mic_server.optics.blank()
    import numpy as np

    # 3) Sum 4 detectors
    n_det = 4
    n_ch_per_det = len(spectrum.data) // n_det
    summed = np.zeros(n_ch_per_det, dtype=float)
    for i in range(n_det):
        s, e = i * n_ch_per_det, (i + 1) * n_ch_per_det
        summed += spectrum.data[s:e]
    spectrum_data = summed

    # 4) Energy calibration
    dispersion, offset = get_dispersion_and_offset(spectrum)
    energy_axis = (np.arange(len(spectrum_data)) * dispersion + offset)/1000 # 1000 for Kev

    # helper: convert energy (keV) → channel index
    def chan_idx(keV: float) -> int:
        return int(np.round(get_channel_index(keV, dispersion, offset)))

    # 5) Integrate Zr peaks (La, Lb, Ka, Kb)
    # zr_peaks_keV = [2.04, 2.12, 15.77, 17.67]
    # window_halfwidth = 30
    # total_counts = 0.0
    # for ek in zr_peaks_keV:
    #     c = chan_idx(ek)
    #     a = max(0, c - window_halfwidth)
    #     b = min(len(spectrum_data), c + window_halfwidth)
    #     total_counts += float(spectrum_data[a:b].sum())
    # total_counts = abs(total_counts - 21000)

    # counts for elements
    import numpy as np
    def get_total_Se_counts(energy_axis, spectrum_data, window=0.05):
        se_peaks = np.array([
            1.379, 1.435,        # L lines
            11.181, 11.222,      # Kα2, Kα1
            12.497               # Kβ1
        ])

        total_counts = 0.0
        for peak in se_peaks:
            mask = (energy_axis >= peak - window) & (energy_axis <= peak + window)
            total_counts += spectrum_data[mask].sum()
        return total_counts
    def get_total_Au_counts(energy_axis, spectrum_data, window=0.05):
        # Known Au peak energies (keV)
        au_peaks = np.array([
            2.123, 2.205, 2.307, 9.713, 11.442, 11.581, 13.382, 13.734
        ])

        total_counts = 0.0
        for peak in au_peaks:
            mask = (energy_axis >= peak - window) & (energy_axis <= peak + window)
            total_counts += spectrum_data[mask].sum()
        return total_counts
    def get_total_S_counts(energy_axis, spectrum_data, window=0.05):
        s_peaks = np.array([
            2.304, 2.307, 2.464     # S Kα₂, Kα₁, Kβ₁
        ])

        total_counts = 0.0
        for peak in s_peaks:
            mask = (energy_axis >= peak - window) & (energy_axis <= peak + window)
            total_counts += spectrum_data[mask].sum()
        return total_counts

    # total_counts = get_total_S_counts(energy_axis, spectrum_data)

    total_counts = get_total_Se_counts(energy_axis, spectrum_data)
    # total_counts = get_total_Au_counts(energy_axis, spectrum_data)

    # 6) Save summed spectrum
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    np.save(f"{out_path}_time_{timestamp}_(x,y)_{(x,y)}disp_{dispersion}_offset_{offset}_4det_summed.npy", spectrum_data)

    print("total counts:", total_counts)
    return float(total_counts)

# def get_eds_black_box(index, indices_all, e1a, e1b, eds_settings, image_size) -> float:
#     """
#     Black box function that returns a target score simulates the blackbox function
#     """

#     e_start,e_end = e1a, e1b
#     # move paused beam to the location index
#     x=int(indices_all[index, 0])/image_size######### TODO: check if x anf y needs to be flipped
#     y=int(indices_all[index, 1])/image_size
#     mic_server.optics.paused_scan_beam_position = list(x, y)#----> (0, 0) should be top left corner 
    
#     # collect eds
#     microscope.optics.unblank()
#     spectrum = mic_server.analysis.eds.acquire_spectrum(eds_settings)
#     microscope.optics.blank()
#     score = spectrum[e_start:e_end].sum()

#     return score




## a) evaluations metrics like nlpd, mse ----
def calculate_mse(y_true, y_pred):
    """Calculate Mean Squared Error (MSE)"""
    #Smaller values indicate better predictions.
    #Squaring ensures that positive and negative errors don't cancel out.
    mse = mean_squared_error(y_true, y_pred)
    return mse

def calculate_nlpd(y_true, y_pred_mean, y_pred_var):
    """Calculate Negative Log Predictive Density (NLPD)"""
    #NLPD evaluates how well the predicted probability distribution matches the true values.
    #Lower NLPD indicates a better match, accounting for both the mean and uncertainty.
    nlpd = 0.5 * torch.log(2 * torch.pi * y_pred_var) + 0.5 * ((y_true - y_pred_mean) ** 2 / y_pred_var)
    return nlpd.mean().item()

### 3d. Utility F:n's - 3

In [ ]:
from tqdm import tqdm
import torch
def calculate_scores_for_patches(unacquired_indices, indices_all, e1a, e1b, black_box_fn = get_eds_black_box, debug = True) -> torch.Tensor:
    """
    Calculate the score for each patch using the black_box function.

    Parameters:
    - patches: Tensor of all data patches.

    Returns:
    - scores: List of scores for each patch.
    """
    scores = []
    for i in unacquired_indices:
        score = black_box_fn(i, indices_all, e1a, e1b)  # Calculate score for each patch
        scores.append(score)
    return torch.tensor(scores)  # Return as a tensor for compatibility

def update_acquired(acquired_data, unacquired_indices, selected_indices, indices_all, e1a, e1b, eds_settings, image_size, black_box_fn = get_eds_black_box) -> (np.array, list):
    for idx in selected_indices:# TODO: It queries the black box everytime on already acquired points:
        acquired_data[idx] = black_box_fn(idx, indices_all, e1a, e1b, eds_settings, image_size)
    unacquired_indices = [idx for idx in unacquired_indices if idx not in selected_indices]


    return acquired_data, unacquired_indices


def load_image_and_features(img: np.ndarray , window_size : int) -> (np.ndarray, np.ndarray):
    coordinates = get_coord_grid(img, step=1, return_dict=False)
    features_all, coords, _ = extract_subimages(img, coordinates, window_size)
    features_all = features_all[:, :, :, 0]
    coords = np.array(coords, dtype=int)
    norm_ = lambda x: (x - x.min()) / np.ptp(x)
    features = norm_(features_all)
    return features, coords# shapes (3366, 5, 5) and (3366, 2)


def prepare_data_from_microscope(window_size: int, haadf: np.ndarray) -> (np.ndarray, np.ndarray):
    global img # TODO: better way to deal with this --> at this point need to plot it when collcting spectrum
    img = haadf
    features, indices_all = load_image_and_features(img, window_size)
    return img, features, indices_all# shapes (55, 70), (3366, 5, 5) and (3366, 2)



def embeddings_and_predictions(model, patches, device="cpu") -> (torch.Tensor, torch.Tensor):
    """
    Get predictions from the trained model
    """
    model.eval()
    patches = patches.to(device)
    with torch.no_grad():
        predictions = model(patches)
        embeddings = model.feature_extractor(patches).view(patches.size(0), -1).cpu().numpy()
    return predictions, embeddings

def train_model(acquired_data, patches, feature_extractor,
                device="cpu", num_epochs=50, log_interval=5,
                scalarizer_zero=False) -> ApproximateGP:
    X_train = torch.stack([patches[idx] for idx in acquired_data]).to(device)
    y_train = torch.tensor(list(acquired_data.values()), dtype=torch.float32).to(device)
    if scalarizer_zero:
        y_train = torch.zeros_like(y_train)

    else:
        # Normalize y_train

        y_train = (y_train - y_train.min()) / (y_train.max() - y_train.min())



    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    inducing_points = X_train[:10]
    model = GPModelDKL(inducing_points=inducing_points, likelihood=likelihood, feature_extractor=feature_extractor).to(device)

    model.train()
    likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for epoch in tqdm(range(1, num_epochs + 1), desc="Training Progress"):
        optimizer.zero_grad()
        output = model(X_train)


        loss = -mll(output, y_train)
        loss.backward()
        optimizer.step()



    return model



### 3e. Bayesian optimization loop

In [ ]:
from stemOrchestrator.acquisition import TFacquisition, DMacquisition
from stemOrchestrator.simulation import DMtwin
from autoscript_tem_microscope_client.enumerations import EdsDetectorType
from stemOrchestrator.process import HAADF_tiff_to_png, tiff_to_png
from autoscript_tem_microscope_client import TemMicroscopeClient
import matplotlib.pyplot as plt
import logging
plot = plt
from typing import Dict
import os

In [ ]:
import torch
from botorch.acquisition import AnalyticAcquisitionFunction
from botorch.utils.transforms import t_batch_mode_transform

class BEACONAcquisitionFunction(AnalyticAcquisitionFunction):
    def __init__(self, model, acquired_data, indices_all, patches, k=10, elite_fraction=0.1, normalize=True):
        """
        BEACON acquisition function for DKL model using Thompson sampling
        
        Args:
            model: Trained DKL model
            acquired_data: Dictionary of {index: value} for acquired points
            indices_all: All coordinate indices
            patches: All image patches
            k: Number of nearest neighbors to consider
            elite_fraction: Fraction of top points to use as elite set
            normalize: Whether to apply z-score normalization
        """
        super().__init__(model=model)
        self.model = model
        self.k = k
        self.normalize = normalize
        
        # Build elite set from acquired data
        if len(acquired_data) > 0:
            # Get top performers (elite set)
            sorted_items = sorted(acquired_data.items(), key=lambda x: x[1], reverse=True)
            n_elite = max(1, int(len(sorted_items) * elite_fraction))
            elite_indices = [idx for idx, _ in sorted_items[:n_elite]]
            
            # Get elite patches and compute posterior means (keep this as means for reference set)
            elite_patches = torch.stack([patches[idx] for idx in elite_indices]).to(model.likelihood.noise.device)
            elite_patches = elite_patches.reshape(-1, 1, patches.shape[-1] * patches.shape[-2])
            
            with torch.no_grad():
                elite_posterior = model.posterior(elite_patches)
                self.elite_behaviors = elite_posterior.mean.detach()  # Keep as means for stable reference
                
                # Normalize if requested
                if self.normalize and self.elite_behaviors.numel() > 1:
                    self.behavior_mean = self.elite_behaviors.mean()
                    self.behavior_std = self.elite_behaviors.std()
                    if self.behavior_std > 1e-8:  # Avoid division by zero
                        self.elite_behaviors = (self.elite_behaviors - self.behavior_mean) / self.behavior_std
        else:
            self.elite_behaviors = torch.empty(0, 1).to(model.likelihood.noise.device)
            self.behavior_mean = 0.0
            self.behavior_std = 1.0
        
    @t_batch_mode_transform(expected_q=1)
    def forward(self, X):
        """Compute BEACON acquisition function value using Thompson sampling"""
        if self.elite_behaviors.numel() == 0:
            return torch.zeros(X.shape[0]).to(X.device)
            
        # Use Thompson sampling for candidate predictions
        with torch.no_grad():
            posterior = self.model.posterior(X)
            # Thompson sampling: sample from the posterior distribution
            candidate_samples = posterior.rsample(torch.Size([1])).squeeze(0)
            
            # Ensure proper shape
            if candidate_samples.dim() == 1:
                candidate_samples = candidate_samples.unsqueeze(1)
                
            # Apply same normalization to candidate samples
            if self.normalize and hasattr(self, 'behavior_std') and self.behavior_std > 1e-8:
                candidate_samples = (candidate_samples - self.behavior_mean) / self.behavior_std
        
        # Calculate distances to elite behaviors (means)
        dist = torch.cdist(candidate_samples, self.elite_behaviors)  # Shape: (batch_size, n_elite)
        
        # Sort distances for each candidate point
        dist_sorted, _ = torch.sort(dist, dim=1)
        
        # Calculate k-NN distances and average them
        n_elite = dist_sorted.size(1)
        k_actual = min(self.k, n_elite)
        
        if k_actual > 0:
            # Take average of k nearest distances
            knn_distances = dist_sorted[:, :k_actual].mean(dim=1)
        else:
            knn_distances = torch.zeros(candidate_samples.size(0)).to(X.device)
        
        return knn_distances.flatten()


In [ ]:


def run(config) -> None:
    # Extract all configuration variables
    ip = config["ip"]
    port = config["port"]
    seed = config["seed"]
    seed_pts = config["seed_pts"]
    budget = config["budget"]
    out_dir_parent = config["out_dir_parent"]
    dataset_name = config["dataset_name"]
    device = config["device"]
    num_epochs = config["num_epochs"]
    normalize_data_flag = config["normalize_data"]
    window_size = config["window_size"]
    scal_stem = config["scal_stem"]
    haadf_exposure = config["haadf_exposure"]
    haadf_resolution = config["haadf_resolution"]
    edx_exposure = config["edx_exposure"]
    no_pre_screen = config["no-pre-screen"]
    use_old_tiff = config["use_old_tiff"]




    scalarizer_zero = False # TODO: deafult value to zero -- so passed to train_model function --> better way to handel

    if scal_stem is not None:## only for pfm: TODO : find better to accomodiate this
        if scal_stem == "sum":
            black_box_fn = get_eds_black_box
            #energy_range to sum in--> TODO: idea to sum for an element like zirconium
            e1a = 0
            e1b = 20 ###### till dispersion?


    else :
        print("what scalarizer you want")


    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    res_dir = Path(out_dir_parent) / f"Dataset_seed{seed}_{dataset_name}_BO_{seed_pts}_epochs{num_epochs}_budget_{budget}_{scal_stem}_ws{window_size}_{timestamp}"
    res_dir.mkdir(parents=True, exist_ok=True)


    # Connect to the microscope server
    global mic_server # TODO: later see better way to do this
    microscope = TemMicroscopeClient()
    microscope.connect(ip, port = port)# 7521 on velox  computer
    mic_server = microscope
    tf_acquisition = TFacquisition(microscope=microscope)


    if use_old_tiff is not None:
        # haadf_tiff_name = "path/of/haadf"
        from stemOrchestrator.process import tiff_to_numpy
        haadf_tiff_name = use_old_tiff
        haadf_np_array = tiff_to_numpy(haadf_tiff_name)
        HAADF_tiff_to_png(haadf_tiff_name)
    
    else:
        # get haadf for dkl
        haadf_np_array, haadf_tiff_name = tf_acquisition.acquire_haadf(exposure = haadf_exposure, resolution=haadf_resolution, folder_path = out_dir_parent)
        HAADF_tiff_to_png(out_dir_parent + haadf_tiff_name)


    
    image_size = haadf_resolution

    # Prepare features and indices from microscope image
    img, features, indices_all = prepare_data_from_microscope(window_size=window_size, haadf=haadf_np_array)


    ############################################
    patches = numpy_to_torch_for_conv(features)
    patches = patches.to(device)

    if normalize_data_flag:
        patches = normalize_data(patches)

    feature_extractor = ConvNetFeatureExtractor(input_channels=1, output_dim=2).to(device)
    acquired_data = {}
    unacquired_indices = list(range(len(indices_all)))####### TODO: need to change later to use the indices_all

    # points to randompy sample from
    selected_indices = random.sample(unacquired_indices, seed_pts)
    seed_indices = selected_indices


    ######### queries microscope to get measuremnt on seed points
    import time
    eds_settings = configure_acquisition(exposure_time=edx_exposure)
    t0 = time.perf_counter()

    acquired_data, unacquired_indices = update_acquired(acquired_data, unacquired_indices, selected_indices, indices_all, e1a, e1b, eds_settings, image_size, black_box_fn=black_box_fn)
    time_hw_seed = time.perf_counter() - t0

    y_train = torch.tensor(list(acquired_data.values()), dtype=torch.float32).to(device)
    y_train = (y_train - y_train.min()) / (y_train.max() - y_train.min())

    print("all the y values for seed points", y_train.cpu())
    plt.close()
    plt.imshow(haadf_np_array)
    plt.scatter(indices_all[seed_indices][:, 1], indices_all[seed_indices][:, 0], s=50, c=y_train.cpu(), cmap = "bwr")
    plt.colorbar()
    plt.savefig(Path(res_dir) / f'_HAADF-pos-seed.png')        
    plt.show()


    from botorch.acquisition import LogExpectedImprovement #ExpectedImprovement
    mean_y_pred_mean_al = []
    mean_y_pred_variance_al = []
    # mae_list = []
    # nlpd_list = []
    # Start Bayesian Optimization loop
    acquired_order = []


    # --- Initialize timing lists before loop ---
    time_hw = []
    time_gp_train = []
    time_acq_opt = []


    for step in range(budget):
        t0 = time.perf_counter()
        # Train the DKL model
        model = train_model(
            acquired_data,
            patches,
            feature_extractor,
            device=device,
            num_epochs=num_epochs,
            scalarizer_zero=scalarizer_zero,
        )
        time_gp_train.append(time.perf_counter() - t0)

        model.eval()

        # Prepare candidate set
        candidate_indices = unacquired_indices

        if no_pre_screen :
            print("No pre-screen")

            X_candidates = torch.stack([patches[idx] for idx in candidate_indices]).to(device)
            X_candidates = X_candidates.reshape(-1, 1, window_size*window_size) # Note this is when using acq f:n directly and not invoking  optimize_acqf_discrete
        else:
            print("uncertainty pre-screen due to compute issues")

            
            max_candidates_for_beacon = 1000
            if len(candidate_indices) > max_candidates_for_beacon:
                print("uncertainty pre-screen due to compute issues")

                # Build all candidates
                X_all = torch.stack([patches[idx] for idx in candidate_indices]).to(device)
                X_all = X_all.reshape(-1, 1, window_size * window_size)

                # Compute predictive variances in chunks
                chunk_size = 512
                var_chunks = []

                with torch.no_grad(), gpytorch.settings.fast_pred_var():
                    for X_chunk in torch.split(X_all, chunk_size, dim=0):
                        posterior_chunk = model.posterior(X_chunk)
                        var_chunk = posterior_chunk.variance

                        # expected shape often: (chunk, 1, 1) -> make it (chunk,)
                        var_chunk = var_chunk.squeeze(-1).squeeze(-1)
                        var_chunks.append(var_chunk)

                all_variances = torch.cat(var_chunks, dim=0)

                # Keep the top most uncertain candidates
                topk = torch.topk(
                    all_variances,
                    k=max_candidates_for_beacon,
                    largest=True,
                ).indices

                # Reduce candidate_indices to top uncertain ones
                candidate_indices = [candidate_indices[i] for i in topk.cpu().tolist()]
            # Build shortlisted candidate tensor
            X_candidates = torch.stack([patches[idx] for idx in candidate_indices]).to(device)
            X_candidates = X_candidates.reshape(-1, 1, window_size * window_size)

        # Before calling BEACON: uncertainty-based pre-screen instead of random sample
        t0 = time.perf_counter()
        acq_values_candidates = None
        if len(acquired_data) > 0:
            beacon_acq_func = BEACONAcquisitionFunction(
                model=model,
                acquired_data=acquired_data,
                indices_all=indices_all,
                patches=patches,
                k=10,
                elite_fraction=0.2,
                normalize=True,
            )

            acq_values_candidates = beacon_acq_func(X_candidates)
            best_idx = torch.argmax(acq_values_candidates).item()
        else:
            best_idx = torch.randint(0, len(candidate_indices), (1,)).item()
        
        time_acq_opt.append(time.perf_counter() - t0)


        selected_index = candidate_indices[best_idx]
        selected_indices = [selected_index]        # Map selected tensors back to indices
        acquired_order.append(int(selected_index))

        # Update acquired data with new observations
        t0 = time.perf_counter()
        acquired_data, unacquired_indices = update_acquired(acquired_data, unacquired_indices, selected_indices, indices_all, e1a, e1b, eds_settings, image_size, black_box_fn=black_box_fn)
        time_hw.append(time.perf_counter() - t0)

        print(f"**************************done BO step {step +1}")


        predictions, embeddings = embeddings_and_predictions(model, patches, device=device)

        y_pred_mean = predictions.mean
        y_pred_var = predictions.variance

        candidate_acq_dict = {candidate_indices[i]: acq_values_candidates[i].item() for i in range(len(candidate_indices))}

        # Calculate MSE and NLPD
        # mse = calculate_mse(true_scalarizer.cpu(), y_pred_mean.cpu())
        # mae = np.sqrt(mse)
        # nlpd = calculate_nlpd(true_scalarizer.cpu(), y_pred_mean.cpu(), y_pred_var.cpu())

        if acq_values_candidates is None:
            print("no plotting as acq_values not triggered yet as len of acquired data not sufficient")
        else:
            print("plotting=========================")
            # Fill the prediction image with predicted mean values
            acq_fn_img = np.zeros((img.shape[0], img.shape[1]))
            y_pred_mean_img = np.zeros((img.shape[0], img.shape[1]))
            y_pred_var_img = np.zeros((img.shape[0], img.shape[1]))
            # Fill the prediction image with predicted mean values
            for j in range(len(indices_all)):
                # Fill acq_fn_img: non-zero only for candidates, zero for acquired points
                if no_pre_screen :
                    if j in candidate_acq_dict:
                        acq_fn_img[indices_all[j][0], indices_all[j][1]] = candidate_acq_dict[j]
                        # #-------> as beacon cannot score for all for high resoltuion ---> many patches ###########
                # else remains 0 (for acquired points)
                
                y_pred_mean_img[indices_all[j][0], indices_all[j][1]] = y_pred_mean[j]
                y_pred_var_img[indices_all[j][0], indices_all[j][1]] = y_pred_var[j]

            # Get predictions for entire dataset (for full image reconstruction)


            # Track surrogate mean and uncertainty over timesteps
            mean_y_pred_mean_al.append(y_pred_mean.mean().cpu())
            mean_y_pred_variance_al.append(y_pred_var.mean().cpu())
            if no_pre_screen:
                # --- Plotting ---
                fig = plt.figure(figsize=(18, 10))
                gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

                ax0 = fig.add_subplot(gs[0, 0])  # HAADF with measurement points
                ax1 = fig.add_subplot(gs[0, 1])  # Predicted mean
                ax2 = fig.add_subplot(gs[0, 2])  # Predicted uncertainty
                ax3 = fig.add_subplot(gs[1, 0])  # Surrogate mean over time
                ax4 = fig.add_subplot(gs[1, 1])  # Surrogate uncertainty over time
                ax5 = fig.add_subplot(gs[1, 2])  # Acquisition function

                # --- 1. HAADF image with measurement points ---
                im0 = ax0.imshow(img, cmap='gray', origin='upper')
                ax0.set_title('HAADF + Measurements')
                fig.colorbar(im0, ax=ax0)

                # Initial seed points in blue
                seed_coords = indices_all[seed_indices]
                ax0.scatter(seed_coords[:, 1], seed_coords[:, 0],
                            color='blue', s=20, label='Seeds', zorder=2)

                # Previous BO steps in green (all except the latest)
                if step > 0:
                    acquired_coords = indices_all[acquired_order[:-1]]
                    ax0.scatter(acquired_coords[:, 1], acquired_coords[:, 0],
                                color='green', s=20, label='Prev BO steps', zorder=2)

                # Current/newest BO point in red (last appended to acquired_order)
                new_coord = indices_all[acquired_order[-1]]
                ax0.scatter(new_coord[1], new_coord[0],
                            color='red', s=80, marker='x', linewidths=1.5,
                            label=f'New point (step {step+1})', zorder=3)

                ax0.legend(loc='upper right', fontsize=7)
                ax0.axis('off')

                # --- 2. Predicted mean ---
                im1 = ax1.imshow(y_pred_mean_img, cmap='viridis', origin='upper')
                ax1.set_title('Predicted Mean')
                fig.colorbar(im1, ax=ax1)
                ax1.axis('off')

                # --- 3. Predicted uncertainty ---
                im2 = ax2.imshow(y_pred_var_img, cmap='viridis', origin='upper')
                ax2.set_title('Predicted Uncertainty')
                fig.colorbar(im2, ax=ax2)
                ax2.axis('off')

                # --- 4. Surrogate mean over timesteps ---
                ax3.plot(range(1, len( mean_y_pred_mean_al) + 1),
                            mean_y_pred_mean_al, marker='o', color='steelblue', linewidth=1.5)
                ax3.set_title('Surrogate Mean over Steps')
                ax3.set_xlabel('Step')
                ax3.set_ylabel('Mean')
                ax3.grid(True, alpha=0.3)

                # --- 5. Surrogate uncertainty over timesteps ---
                ax4.plot(range(1, len(mean_y_pred_variance_al) + 1),
                            mean_y_pred_variance_al, marker='o', color='darkorange', linewidth=1.5)
                ax4.set_title('Surrogate Uncertainty over Steps')
                ax4.set_xlabel('Step')
                ax4.set_ylabel('Variance')
                ax4.grid(True, alpha=0.3)

                # --- 6. Acquisition function ---
                im5 = ax5.imshow(acq_fn_img, cmap='viridis', origin='upper')
                ax5.set_title('Acquisition Function')
                fig.colorbar(im5, ax=ax5)
                # Mark the selected point on acq fn map too
                ax5.scatter(new_coord[1], new_coord[0],
                            color='red', s=80, marker='x', linewidths=1.5,
                            label=f'Selected', zorder=3)
                ax5.legend(loc='upper right', fontsize=7)
                ax5.axis('off')

                plt.suptitle(f'BO Step {step + 1} with Beacon', fontsize=14)
                plt.savefig(Path(res_dir) / f'_BO_step{step}.png', dpi=150, bbox_inches='tight')
                plt.show()
                plt.close()


                # Save predictions as a .pkl file
                predictions_data = {
                    "acq_fn_img": acq_fn_img ,
                    "y_pred_mean_img": y_pred_mean_img,
                    "y_pred_var_img": y_pred_var_img,

                    "embeddings": embeddings,
                }


                with open(Path(res_dir) / f'predictions_BO_step{step}.pkl', 'wb') as f:
                    pickle.dump(predictions_data, f)
            else :

                # --- Plotting ---
                fig = plt.figure(figsize=(15, 10))
                gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

                ax0 = fig.add_subplot(gs[0, 0])  # HAADF with measurement points
                ax1 = fig.add_subplot(gs[0, 1])  # Predicted mean
                ax2 = fig.add_subplot(gs[0, 2])  # Predicted uncertainty
                ax3 = fig.add_subplot(gs[1, 0])  # Surrogate mean over time
                ax4 = fig.add_subplot(gs[1, 1])  # Surrogate uncertainty over time

                # --- 1. HAADF image with measurement points ---
                im0 = ax0.imshow(img, cmap='gray', origin='upper')
                ax0.set_title('HAADF + Measurements')
                fig.colorbar(im0, ax=ax0)

                # Initial seed points in blue
                seed_coords = indices_all[seed_indices]
                ax0.scatter(seed_coords[:, 1], seed_coords[:, 0],
                            color='blue', s=20, label='Seeds', zorder=2)

                # Previous BO steps in green (all except the latest)
                if step > 0:
                    acquired_coords = indices_all[acquired_order[:-1]]
                    ax0.scatter(acquired_coords[:, 1], acquired_coords[:, 0],
                                color='green', s=20, label='Prev BO steps', zorder=2)

                # Current/newest BO point in red (last appended to acquired_order)
                new_coord = indices_all[acquired_order[-1]]
                ax0.scatter(new_coord[1], new_coord[0],
                            color='red', s=80, marker='x', linewidths=1.5,
                            label=f'New point (step {step+1})', zorder=3)

                ax0.legend(loc='upper right', fontsize=7)
                ax0.axis('off')

                # --- 2. Predicted mean ---
                im1 = ax1.imshow(y_pred_mean_img, cmap='viridis', origin='upper')
                ax1.set_title('Predicted Mean')
                fig.colorbar(im1, ax=ax1)
                ax1.axis('off')

                # --- 3. Predicted uncertainty ---
                im2 = ax2.imshow(y_pred_var_img, cmap='viridis', origin='upper')
                ax2.set_title('Predicted Uncertainty')
                fig.colorbar(im2, ax=ax2)
                ax2.axis('off')

                # --- 4. Surrogate mean over timesteps ---
                ax3.plot(range(1, len( mean_y_pred_mean_al) + 1),
                        mean_y_pred_mean_al, marker='o', color='steelblue', linewidth=1.5)
                ax3.set_title('Surrogate Mean over Steps')
                ax3.set_xlabel('Step')
                ax3.set_ylabel('Mean')
                ax3.grid(True, alpha=0.3)

                # --- 5. Surrogate uncertainty over timesteps ---
                ax4.plot(range(1, len(mean_y_pred_variance_al) + 1),
                        mean_y_pred_variance_al, marker='o', color='darkorange', linewidth=1.5)
                ax4.set_title('Surrogate Uncertainty over Steps')
                ax4.set_xlabel('Step')
                ax4.set_ylabel('Variance')
                ax4.grid(True, alpha=0.3)

                plt.suptitle(f'BO Step {step} with Novelty in Target space', fontsize=14)
                plt.savefig(Path(res_dir) / f'_BO_step{step}.png', dpi=150, bbox_inches='tight')
                plt.show()
                plt.close()


                # Save predictions as a .pkl file
                predictions_data = {
                    # "acq_fn_img": acq_fn_img ,
                    "y_pred_mean_img": y_pred_mean_img,
                    "y_pred_var_img": y_pred_var_img,

                    "embeddings": embeddings,
                }


                with open(Path(res_dir) / f'predictions_BO_step{step}.pkl', 'wb') as f:
                    pickle.dump(predictions_data, f)
            





            # imshow 4 images: img, pred_mean_img, pred_var_img, true_scal_img


    # Save predictions as a .pkl file
    Active_learning_statistics = {
        "time_hw_seed": time_hw_seed,
        "time_hw": np.array(time_hw),
        "time_gp_train": np.array(time_gp_train),
        "time_acq_opt": np.array(time_acq_opt),
        "acquired_order": acquired_order,
        "img": img,
        "features": features,
        "indices_all": np.array(indices_all),
        "seed_indices": np.array(seed_indices),
        "unacquired_indices": np.array(unacquired_indices),
        "mean_y_pred_mean_al": np.array(mean_y_pred_mean_al),
        "mean_y_pred_variance_al": np.array(mean_y_pred_variance_al),
        # "mae": np.array(mae_list),
        # "nlpd": np.array(nlpd_list)
                }

    with open(Path(res_dir) / f'Active_learning_statistics.pkl', 'wb') as f:
        pickle.dump(Active_learning_statistics, f)


    ##############################

    predictions_data = Active_learning_statistics
    # Extract necessary data
    img = np.array(predictions_data["img"])  # Image or grid for background visualization
    seed_indices = np.array(predictions_data["seed_indices"])  # Initial sampled indices (referring to positions in indices_all)
    unacquired_indices = np.array(predictions_data["unacquired_indices"])  # Remaining indices
    # indices_all = np.array(predictions_data["indices_all"])  # All possible indices (coordinates)

    # Map seed_indices and unacquired_indices to their coordinates in indices_all
    seed_coords = indices_all[seed_indices]
    unacquired_coords = indices_all[unacquired_indices]

    indices_all = np.array(predictions_data["indices_all"])
    acquired_indices = np.array(predictions_data["acquired_order"])   # TRUE sequence
    acquired_coords = indices_all[acquired_indices]
    time_order = np.arange(len(acquired_coords))
    
    # Plot the results
    plt.figure(figsize=(10, 8))

    # Display the image or grid as the background
    plt.imshow(img, cmap="gray", origin="upper")

    # Plot the seed points in blue
    plt.scatter(seed_coords[:, 1], seed_coords[:, 0], c="b", label="Seed Points", marker="o")

    # time_order = np.arange(len(acquired_coords))  # Create a sequence representing time
    scatter = plt.scatter(acquired_coords[:, 1], acquired_coords[:, 0], c=time_order, cmap="bwr", label="Acquired Points", marker="x")

    # Plot the unacquired points in green
    # plt.scatter(unacquired_coords[:, 1], unacquired_coords[:, 0], c="g", label="Unacquired Points", marker="+")

    # Set plot labels and legend
    plt.xlabel("X-axis")
    plt.ylabel("Y-axis")
    plt.title("Active Learning Trajectory")
    plt.legend()
    plt.grid(True)
    # Add a colorbar and label it as "Steps"
    cbar = plt.colorbar(scatter)
    cbar.set_label("Steps")


    plt.savefig(Path(res_dir) / "AL_traj.png")
    plt.show()
    plt.close()


    # Extract data for learning curve
    mean_y_pred_mean_al = np.array(predictions_data["mean_y_pred_mean_al"])
    mean_y_pred_variance_al = np.array(predictions_data["mean_y_pred_variance_al"])
    # mae_list = np.array(predictions_data["mae"])
    # nlpd_list = np.array(predictions_data["nlpd"])

    steps = np.arange(len(mean_y_pred_mean_al))  # Assuming the steps are sequential indices

    # Calculate the upper and lower bounds using variance
    upper_bound = mean_y_pred_mean_al + np.sqrt(mean_y_pred_variance_al)
    lower_bound = mean_y_pred_mean_al - np.sqrt(mean_y_pred_variance_al)

    # Plot the learning curve
    plt.figure(figsize=(10, 6))

    # Plot the mean predictions
    plt.plot(steps, mean_y_pred_mean_al, label="Mean Prediction", color="blue", linewidth=2)

    # Fill between the upper and lower bounds to represent variance
    plt.fill_between(
        steps,
        lower_bound,
        upper_bound,
        color="blue",
        alpha=0.2,
        label="Variance (±1 std)"
    )

    # Add labels, title, and legend
    plt.xlabel("Steps")
    plt.ylabel("Mean Prediction")
    plt.title("Learning Curve with Variance")
    plt.legend()
    plt.grid(True)
    plt.savefig(Path(res_dir) /"AL_learning_curve.png")
    plt.show()
    plt.close()


    steps = range(1, len(time_hw) + 1)

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.stackplot(steps,
                time_hw,
                time_gp_train,
                time_acq_opt,
                labels=['Microscope (HW)', 'GP Training', 'Acquisition Opt'],
                colors=['steelblue', 'gold', 'darkorange'],
                alpha=0.8)

    ax.set_xlabel('BO Step')
    ax.set_ylabel('Time (seconds)')
    ax.set_title('Timing per BO Step')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.2)

    # Annotate seed acquisition time as text
    ax.text(0.98, 0.98, f'Seed HW time: {time_hw_seed:.2f}s',
            transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    plt.tight_layout()
    plt.savefig(Path(res_dir) / 'timing_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


    # plt.figure(figsize=(10, 6))

    # plt.plot(steps, mae_list, color="red", linewidth=2)

    # plt.xlabel("Steps")
    # plt.ylabel("Mean absolute ERROR")
    # plt.grid(True)
    # plt.savefig(Path(res_dir) / "AL_error_curve.png")
    # plt.show()
    # plt.close()


### 3f. Set parameters and Run experiments

In [ ]:
from stemOrchestrator.logging_config   import setup_logging
from datetime import datetime
import os


In [ ]:
from stemOrchestrator.acquisition import TFacquisition, DMacquisition
from stemOrchestrator.simulation import DMtwin
from stemOrchestrator.process import HAADF_tiff_to_png, tiff_to_png
from autoscript_tem_microscope_client import TemMicroscopeClient
import matplotlib.pyplot as plt
import logging
plot = plt
from typing import Dict

In [11]:
import os
import json
from pathlib import Path

ip = os.getenv("MICROSCOPE_IP")
port = os.getenv("MICROSCOPE_PORT")

if not ip or not port:
    secret_path = Path("../../../../config_secret.json")
    if secret_path.exists():
        with open(secret_path, "r") as f:
            secret = json.load(f)
            ip = ip or secret.get("ip_TF")
            port = port or secret.get("port_TF")
print(ip, port)

current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_name = "edx-haadf-beacon"
data_folder  = f"./{exp_name}_{current_time}/"
os.makedirs(name = data_folder, exist_ok=True)
out_path = data_folder
setup_logging(out_path=out_path) 


config = {
        "ip": ip,
        "port": port,
        "haadf_exposure": 10e-6,  # micro-seconds per pixel
        "haadf_resolution": 64, # square
        "edx_exposure": 2, # seconds
        "seed": 5,
        "seed_pts" : 10, # How many points you want to start your BO with?
        "budget" : 100, # How many experimental budget you have?
        "out_dir_parent": out_path, # recommended : leave as is
        "dataset_name": "live_mic", # name of data to be loaded in DTmicroscope
        "device": "cuda",
        "num_epochs": 100, # Number of epoch the dkl model trains at each experimental step - Might need tuning based on data
        "normalize_data": True, 
        "window_size": 5, # For square patches - structure property relationship
        "scal_stem": "sum", # What physics interested in? options on this data: "loop_area", "loop_height", "positive_nucleation_bias", "negative_nucleation_bias"
        "no-pre-screen": True,
        "use_old_tiff": None,
        }
run(config)

KeyboardInterrupt: 